# DMI quick example notebook

This notebook shows a minimal workflow to run the Dipole Mode Index (DMI) diagnostic and generate index and regression plots.

The only required imports are `DMI` and `PlotDMI` from `aqua.diagnostics.teleconnections`.

In [ ]:
from aqua.diagnostics.teleconnections import DMI, PlotDMI

## Define dataset and reference

As in the ENSO notebook, we define one model dataset and one reference (ERA5).

In [ ]:
dataset_dict = {
    "catalog": "climatedt-phase1",
    "model": "IFS-NEMO",
    "exp": "historical-1990",
    "source": "lra-r100-monthly",
}
obs_dict = {
    "catalog": "obs",
    "model": "ERA5",
    "exp": "era5",
    "source": "monthly",
}
common_dict = {"loglevel": "INFO"}

In [ ]:
dmi_dataset = DMI(**dataset_dict, **common_dict)
dmi_obs = DMI(**obs_dict, **common_dict)

## Retrieve data and compute index

DMI computes anomalies over two Indian Ocean regions and then takes their difference.

In [ ]:
dmi_dataset.retrieve()
dmi_obs.retrieve()

dmi_dataset.compute_index()
dmi_obs.compute_index()

dmi_dataset.index
dmi_obs.index

In [ ]:
dmi_dataset.save_netcdf(dmi_dataset.index, diagnostic="dmi", diagnostic_product="index")
dmi_obs.save_netcdf(dmi_obs.index, diagnostic="dmi", diagnostic_product="index")

## Regression maps

By default, regression is computed on the same variable used for the index (`tos`).

In [ ]:
reg_data = dmi_dataset.compute_regression(season="annual")
reg_obs = dmi_obs.compute_regression(season="annual")

dmi_dataset.save_netcdf(reg_data, diagnostic="dmi", diagnostic_product="regression")
dmi_obs.save_netcdf(reg_obs, diagnostic="dmi", diagnostic_product="regression")

## Plot index and regression

Load data before plotting maps to avoid cartopy+dask issues.

In [ ]:
plot = PlotDMI(indexes=dmi_dataset.index, ref_indexes=dmi_obs.index, loglevel="INFO")

fig_index, _ = plot.plot_index()
description_index = plot.set_index_description()
plot.save_plot(fig_index, diagnostic_product="index", metadata={"description": description_index})

In [ ]:
reg_data.load()
reg_obs.load()

fig_reg = plot.plot_maps(maps=[reg_data], ref_maps=[reg_obs], statistic="regression")
description_reg = plot.set_map_description(maps=[reg_data], ref_maps=[reg_obs], statistic="regression")
plot.save_plot(fig_reg, diagnostic_product="regression", metadata={"description": description_reg})